In [3]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sn

In [4]:
from pathlib import Path
files = sorted(Path('../data').glob('*.csv'))   # check what this finds
len(files)

frames = []
for f in files:
    frames.append(pd.read_csv(f))  # read, strip column names, append to frames
df = pd.concat(frames, ignore_index=True)

df.shape

(2830743, 79)

In [5]:
df.shape

(2830743, 79)

In [6]:
df.columns

Index([' Destination Port', ' Flow Duration', ' Total Fwd Packets',
       ' Total Backward Packets', 'Total Length of Fwd Packets',
       ' Total Length of Bwd Packets', ' Fwd Packet Length Max',
       ' Fwd Packet Length Min', ' Fwd Packet Length Mean',
       ' Fwd Packet Length Std', 'Bwd Packet Length Max',
       ' Bwd Packet Length Min', ' Bwd Packet Length Mean',
       ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s',
       ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min',
       'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max',
       ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std',
       ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags',
       ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length',
       ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s',
       ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean',
       ' Packet Length Std', ' Packet Length Variance', '

In [7]:
def data_cleaning(df):
    df.columns=df.columns.str.strip()

    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna()

    df = df.drop(columns=['Fwd Header Length.1'])

    df = df.drop_duplicates()


    return df


In [8]:
df = data_cleaning(df)

In [9]:

df.columns
df.shape

(2520798, 78)

In [10]:
print(df.shape)
print(df['Label'].nunique())

(2520798, 78)
15


In [11]:
df['Label'] = df['Label'].str.replace(r'[^\x00-\x7F]+', '-', regex=True)
print(df['Label'].value_counts())

Label
BENIGN                        2095057
DoS Hulk                       172846
DDoS                           128014
PortScan                        90694
DoS GoldenEye                   10286
FTP-Patator                      5931
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1948
Web Attack - Brute Force         1470
Web Attack - XSS                  652
Infiltration                       36
Web Attack - Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [12]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Label'])
labels = df['Label']

X_train, X_test, lab_train, lab_test = train_test_split(
    X, labels, test_size=0.2, random_state=42, stratify=labels)

y_train = (lab_train != 'BENIGN').astype(int)
y_test = (lab_test != 'BENIGN').astype(int)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(2016638, 77) (504160, 77)
0.1688914916807082 0.1688908283084735


In [13]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
print(dummy.score(X_test, y_test))

0.8311091716915265


In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [15]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)
y_pred = lr.predict(X_test_s)

In [16]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=4))

[[409149   9863]
 [ 11753  73395]]
              precision    recall  f1-score   support

           0     0.9721    0.9765    0.9743    419012
           1     0.8815    0.8620    0.8716     85148

    accuracy                         0.9571    504160
   macro avg     0.9268    0.9192    0.9230    504160
weighted avg     0.9568    0.9571    0.9569    504160



In [17]:
results = pd.DataFrame({'label': lab_test.values, 'predicted_attack': y_pred})
print(results.groupby('label')['predicted_attack'].agg(['mean', 'count']))

                                mean   count
label                                       
BENIGN                      0.023539  419012
Bot                         0.248718     390
DDoS                        0.854353   25603
DoS GoldenEye               0.819154    2057
DoS Hulk                    0.898869   34569
DoS Slowhttptest            0.751434    1046
DoS slowloris               0.875580    1077
FTP-Patator                 0.685497    1186
Heartbleed                  0.500000       2
Infiltration                0.285714       7
PortScan                    0.888417   18139
SSH-Patator                 0.009317     644
Web Attack - Brute Force    0.000000     294
Web Attack - Sql Injection  0.000000       4
Web Attack - XSS            0.000000     130


In [18]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf.fit(X_train_s, y_train)
y_pred_rf = rf.predict(X_test_s)

print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf, digits=4))

/Users/krish.sawant/Downloads/NIDS/.venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/krish.sawant/Downloads/NIDS/.venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/krish.sawant/Downloads/NIDS/.venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/krish.sawant

[[418744    268]
 [   252  84896]]
              precision    recall  f1-score   support

           0     0.9994    0.9994    0.9994    419012
           1     0.9969    0.9970    0.9969     85148

    accuracy                         0.9990    504160
   macro avg     0.9981    0.9982    0.9982    504160
weighted avg     0.9990    0.9990    0.9990    504160



In [21]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                    n_jobs=-1, random_state=42)
xgb.fit(X_train_s, y_train)
y_pred_xgb = xgb.predict(X_test_s)

print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb, digits=4))

[[418734    278]
 [   138  85010]]
              precision    recall  f1-score   support

           0     0.9997    0.9993    0.9995    419012
           1     0.9967    0.9984    0.9976     85148

    accuracy                         0.9992    504160
   macro avg     0.9982    0.9989    0.9985    504160
weighted avg     0.9992    0.9992    0.9992    504160

